# DHR-666 DP3 DDECal Python Binding

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
#set the path to the notebooks folder so that we can import modules from there (TODO : DONE )
#sys.path.insert(0, "/home/maneesh/Work/SKAO/ska-sdp-instrumental-calibration/notebooks")

sys.path.insert(0, "/home/ska/Desktop/SKAO/git_clone_ska_sdp_func_python/final_install/ska-sdp-instrumental-calibration/notebooks")

In [ ]:
# Modify as required
#set the path to the measurement set (MS) file (TODO : DONE)
#ms_path = "/home/ska/Work/data/INST/lg3/cal_bpp_vis-lg3-rotated.small.ms"
ms_path = "/home/ska/Desktop/SKAO/git_clone_ska_sdp_func_python/final_install/ska-sdp-instrumental-calibration/corrected_avg_vis_1.ms"
#set the path to the DP3 skymodel file(DONE)
#dp3_skymodel_path = "./dp3_skymodel.txt"
dp3_skymodel_path = "/home/ska/Desktop/SKAO/git_clone_ska_sdp_func_python/final_install/ska-sdp-instrumental-calibration/notebooks/inst_independent_run/sky/corrected_avg_vis_1/sky_model.csv"

#set the path to the GLEAM skymodel file(TODO : DONE)
#gleam_skymodel_path = "/home/ska/Work/data/INST/sim/gleamegc.dat"
gleam_skymodel_path  = "/home/ska/Desktop/SKAO/git_clone_ska_sdp_func_python/final_install/ska-sdp-instrumental-calibration/tests/e2e/resources/gleamdata.dat")

#set the output paths   (TODO : DONE)
inst_output_path = "./inst_output_path"
dp3_output_path = "./dp3_execute_output"
#set the path to the cache directory (TODO : DONE)
#inst_cache_dir = "/home/maneesh/Work/SKAO/ska-sdp-instrumental-calibration/cache"
inst_cache_dir = "/home/ska/Desktop/SKAO/git_clone_ska_sdp_func_python/final_install/ska-sdp-instrumental-calibration/cache"

In [ ]:
import os
import shutil
import dp3
import numpy as np
import xarray as xr
import tempfile
import h5py
import dask
from distributed import Client

from notebook_utils import compare_arrays

from ska_sdp_func_python.visibility.operations import expand_polarizations
from ska_sdp_datamodels.calibration import GainTable
from ska_sdp_datamodels.visibility import Visibility

from ska_sdp_instrumental_calibration.data_managers.solution_interval import (
    SolutionIntervals,
)
from ska_sdp_instrumental_calibration.numpy_processors.solvers.solver import Solver
from ska_sdp_instrumental_calibration.xarray_processors.solver import run_solver
from ska_sdp_instrumental_calibration.data_managers.gaintable import create_gaintable_from_h5parm
from ska_sdp_instrumental_calibration.data_managers.schema.polarization import jones_to_pol, pol_to_jones #(TODO : Ask Maneesh)

from ska_sdp_instrumental_calibration.stages.load_data import load_data_stage
from ska_sdp_instrumental_calibration.stages.predict_visibilities import predict_visibilities
from ska_sdp_instrumental_calibration.stages.bandpass_calibration import bandpass_calibration_stage
from ska_sdp_instrumental_calibration.scheduler import UpstreamOutput

In [ ]:
#set the path to the DP3 server ( TODO : DONE)
client = Client("tcp://127.0.0.1:34567")

In [ ]:
def clear_and_create_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.mkdir(path)

for path in [inst_output_path, dp3_output_path]:
    clear_and_create_dir(path)

In [ ]:
#set the paths for the DP3 output files
parset_file = f"{dp3_output_path}/gaincal.parset"
#set the path for the DP3 output measurement set (MS) file
msout_path = f"{dp3_output_path}/dp3_output.ms"
#set the path for the DP3 output gain table file
dp3_output_gainpath = f"{dp3_output_path}/gaincal_via_execute.h5parm"

## DP3 via execute

In [ ]:

#generate the parset file for DP3 gaincal (ask AI, TODO : DONE)
#parset_content = f"""
#msin = "{ms_path}"
#msin.datacolumn = DATA
#msin.chunksize = 500
#msout = "{msout_path}"

#steps = [cal]

#cal.type = gaincal
#cal.sourcedb = "{dp3_skymodel_path}"
#cal.parmdb = "{dp3_output_gainpath}"
#cal.caltype = fulljones
#cal.solint = 0
#cal.nchan = 1
#"""

#with open(parset_file, "w") as file:
#    file.write(parset_content)

# Generate the parset file for DP3 ddecal

parset_content = f"""
msin = "{ms_path}"
msin.datacolumn = DATA
msin.chunksize = 500
msout = "{msout_path}"

steps = [cal]

cal.type = ddecal
cal.sourcedb = "{dp3_skymodel_path}"
cal.parmdb = "{dp3_output_gainpath}"
cal.caltype = fulljones
cal.solint = 0
cal.nchan = 1
"""

with open(parset_file, "w") as file:
    file.write(parset_content)

print(f"DP3 ddecal parset written to: {parset_file}")

In [ ]:
# this cell is for running (TODO  : Ask Maneesh if this is the correct way to run DP3 ddecal)
import subprocess

# 1. Copy the system's current environment variables
custom_env = os.environ.copy()
custom_env["OPENBLAS_NUM_THREADS"]="1"

subprocess.run(
    ["DP3.py",
    parset_file,],
    env=custom_env
)

## DP3 via in-memory python bindings 

In [ ]:
#dp3_gaincal_solve is a function that solves for the gain calibration using DP3. It takes in the measurement set path, the skymodel path, 
#and the output gain table path as arguments. It returns a DP3ObservationInfo object that contains information about the observation.
#Change dp3_gaincal_solve to dp3_ddecal_solve(TODO : DONE) generate via AI
#from ska_sdp_instrumental_calibration.numpy_processors.solvers.dp3_solvers import dp3_gaincal_solve, DP3ObservationInfo
from ska_sdp_instrumental_calibration.numpy_processors.solvers.dp3_ddecal_solvers import dp3_ddecal_solve, DP3ObservationInfo #(TODO : Ask Maneesh DONE)

### Computation (without modelvis)

In [ ]:
up_out = UpstreamOutput()
input_ms_as_list = [ms_path]

up_out = load_data_stage(up_out, inst_output_path, input_ms_as_list, cache_directory=inst_cache_dir)

# Set just for convinience of running further cells
input_vis = up_out[0].vis
init_gaintable = up_out[0].gaintable

In [ ]:
input_vis_computed = input_vis.compute()
init_gaintable_computed = init_gaintable.compute()

In [ ]:
#Get this from AI (TODO : Ask Maneesh)
#gain, _weight, _residual = dp3_gaincal_solve(
#    input_vis_computed.vis.data,
#    input_vis_computed.flags.data,
#    input_vis_computed.weight.data,
#    None,
#    None,
#    pol_to_jones(init_gaintable_computed.CALPARAM_GAIN.data),
#    pol_to_jones(init_gaintable_computed.CALPARAM_WEIGHT.data),
#    pol_to_jones(init_gaintable_computed.CALPARAM_RESIDUAL.data),
#    input_vis_computed.antenna1.data,
#    input_vis_computed.antenna2.data,
#    caltype="fulljones",
#    skymodel_path=dp3_skymodel_path,
#    observation=DP3ObservationInfo.from_visibility(input_vis),
#)


from ska_sdp_instrumental_calibration.numpy_processors.solvers.dp3_ddecal_solvers import dp3_ddecal_solve, DP3ObservationInfo

ddecal_h5parm = dp3_ddecal_solve(
    input_vis_computed.vis.data,
    input_vis_computed.flags.data,
    input_vis_computed.weight.data,
    input_vis_computed.antenna1.data,
    input_vis_computed.antenna2.data,
    skymodel_path=dp3_skymodel_path,
    observation=DP3ObservationInfo.from_visibility(input_vis),
    output_h5parm_path=dp3_output_h5parm_path,
    mode="fulljones",
    solint=0,
    nchan=1,
    niter=50,
    tol=1e-5,
)


In [ ]:
#should be ddecal everywhere(TODO : DONE) just change the name of the function to dp3_ddecal_solve(TODO : DONE) generate via AI
#dp3_gaincal_gaintable = init_gaintable.copy(deep=True)

#dp3_gaincal_gaintable.CALPARAM_GAIN.data = jones_to_pol(_gain)
#dp3_gaincal_gaintable.CALPARAM_WEIGHT.data = jones_to_pol(_weight)
#dp3_gaincal_gaintable.CALPARAM_RESIDUAL.data = jones_to_pol(_residual)

dp3_ddecal_gaintable = init_gaintable.copy(deep=True)

dp3_ddecal_gaintable.CALPARAM_GAIN.data = jones_to_pol(_gain)  #TODO : Ask Maneesh
dp3_ddecal_gaintable.CALPARAM_WEIGHT.data = jones_to_pol(_weight)
dp3_ddecal_gaintable.CALPARAM_RESIDUAL.data = jones_to_pol(_residual)

## Comparision (without modelvis)

In [ ]:
dp3_gaintable: GainTable = create_gaintable_from_h5parm(
    dp3_output_gainpath,
    interval=SolutionIntervals(input_vis.time.data, "full").intervals,
    vis=input_vis,
)

In [ ]:
#xr.testing.assert_allclose(dp3_gaincal_gaintable.CALPARAM_GAIN, dp3_gaintable.CALPARAM_GAIN) ( TODO : Do till here)
xr.testing.assert_allclose(dp3_ddecal_gaintable.CALPARAM_GAIN, dp3_gaintable.CALPARAM_GAIN)

## Calling Solver in Dask Distribution

In [ ]:
#ddecal solver is to be written Dp3DDECalSolver (TODO) generate via AI
from ska_sdp_instrumental_calibration.numpy_processors.solvers.dp3_solvers import Dp3GaincalSolver

## Aligned comparison: `run_solver` vs DP3 execute

`solver_gaintable` above uses INST's `predict_vis` model, while DP3 execute predicts its own model from `dp3_skymodel.txt`, so the two gaintables are not directly comparable.

To align them, we use a common model:
1. Copy the input MS and use DP3 `predict` to write the model into its `MODEL_DATA` column.
2. DP3 execute gaincal reads that column (`cal.usemodelcolumn` + `cal.modelcolumn`).
3. INST loads `DATA` and `MODEL_DATA` from the same MS as `vis` / `modelvis` for `run_solver`.

Both solvers then see identical data, model, flags and weights.

In [ ]:
# NOTE: `output_path` is overwritten by the INST cell above,
# so derive the DP3 output dir from the parset path.
aligned_dir = "./dp3_aligned_execute_output"
aligned_ms_path = f"{aligned_dir}/aligned.ms"
aligned_dp3_gainpath = f"{aligned_dir}/gaincal_via_execute.h5parm"

clear_and_create_dir(aligned_dir)

# Work on a copy so that the input MS is never modified
shutil.copytree(ms_path, aligned_ms_path)

# Step 1: DP3 predict -> MODEL_DATA column of the copied MS (in place)
predict_parset = f"""
msin = "{aligned_ms_path}"
msin.datacolumn = DATA
msout = .
msout.datacolumn = MODEL_DATA

steps = [predict]

predict.type = predict
predict.sourcedb = "{dp3_skymodel_path}"
"""

predict_parset_file = f"{aligned_dir}/predict.parset"
with open(predict_parset_file, "w") as file:
    file.write(predict_parset)

dp3.execute(predict_parset_file, [])

In [ ]:
# Step 2: DP3 gaincal via execute, reading the model from MODEL_DATA (# Here it should be DDECal(TODO : DONE), also change the parameters below)
# instead of predicting it from the sourcedb.
# niter / tol are set to match the Solver used below.
aligned_ddecal_parset = f"""
msin = "{aligned_ms_path}"
msin.datacolumn = DATA
msout = ""

steps = [cal]

cal.type = ddecal
cal.usemodelcolumn = true
cal.modelcolumn = MODEL_DATA
cal.parmdb = "{aligned_dp3_gainpath}"
cal.caltype = fulljones
cal.solint = 0
cal.nchan = 1
cal.maxiter = 50
cal.tolerance = 1e-6
"""

aligned_ddecal_parset_file = f"{aligned_dir}/ddecal.parset"
with open(aligned_ddecal_parset_file, "w") as file:
    file.write(aligned_ddecal_parset)

dp3.execute(aligned_ddecal_parset_file, [])

In [ ]:
# INST: read DATA and MODEL_DATA of the same MS as two Visibility datasets.
# NOTE: the zarr cache key is "<ms>_fid<id>_ddid<id>" and does NOT include
# the datacolumn, so each column needs its own cache_directory.
aligned_up_out = load_data_stage(
    UpstreamOutput(),
    aligned_dir,
    [aligned_ms_path],
    datacolumn="DATA",
    cache_directory=f"{aligned_dir}/cache_data",
)[0]
aligned_model_up_out = load_data_stage(
    UpstreamOutput(),
    aligned_dir,
    [aligned_ms_path],
    datacolumn="MODEL_DATA",
    cache_directory=f"{aligned_dir}/cache_model",
)[0]

aligned_vis = aligned_up_out.vis
aligned_modelvis = aligned_model_up_out.vis

In [ ]:
# # Optional diagnostic: how far is INST's predicted model from DP3's model?
# # Non-zero differences here explain why the un-aligned comparison fails.

# def dump():
#     predict_up_out = predict_visibilities(up_out[0], inst_output_path, input_ms_as_list, gleamfile=gleam_skymodel_path, use_everybeam=False, fov=0.1)

#     compare_arrays(
#         predict_up_out.modelvis.vis.data[... , 3],
#         aligned_modelvis.vis.data[... , 3],
#         rtol=1e-5,
#         atol=1e-8,
#         meta="Modelvis (INST predict vs DP3 predict)",
#     )

# dump()

In [ ]:
#make it dp3_ddecal
solver = Solver.get_solver(
    "dp3_ddecal", crosspol=True, niter=50, tol=1e-6
)

In [ ]:
aligned_solver_gaintable = run_solver(
    aligned_vis,
    aligned_modelvis,
    aligned_up_out.gaintable,
    solver,
)

aligned_solver_gaintable_computed = dask.compute(
    aligned_solver_gaintable, optimize_graph=True
)[0]

In [ ]:
aligned_dp3_gaintable: GainTable = create_gaintable_from_h5parm(
    aligned_dp3_gainpath,
    interval=SolutionIntervals(aligned_vis.time.data, "full").intervals,
    vis=aligned_vis,
)

In [ ]:
compare_arrays(
    aligned_solver_gaintable_computed.CALPARAM_GAIN.data,
    aligned_dp3_gaintable.CALPARAM_GAIN.data,
    meta="Gain (solver vs DP3 execute)",
)

In [ ]:
compare_arrays(
    aligned_solver_gaintable_computed.CALPARAM_WEIGHT.data,
    aligned_dp3_gaintable.CALPARAM_WEIGHT.data,
    meta="Weight (solver vs DP3 execute)",
)

## Running INST independently

For testing with larger dataset. Not running DP3 for comparision here

In [ ]:
from ska_sdp_instrumental_calibration.stages.configuration_models import (
    PlotConfig,
    RunSolverConfig,
    VisibilityFilterConfig,
)

def t():
    _ms_path = "/home/ska/Work/data/INST/lg3/cal_bpp_vis-lg3-rotated.ms"
    _inst_output_path = "./inst_independent_run/"
    clear_and_create_dir(_inst_output_path)

    _inst_up_out_ = UpstreamOutput()
    _input_ms_as_list = [_ms_path]
    _inst_up_out_ = load_data_stage(_inst_up_out_, _inst_output_path, _input_ms_as_list, cache_directory=inst_cache_dir)

    _predict_up_out = predict_visibilities(_inst_up_out_[0], _inst_output_path, _input_ms_as_list, gleamfile=gleam_skymodel_path, use_everybeam=False, fov=0.1)

    _up_out = bandpass_calibration_stage(_predict_up_out, _inst_output_path, RunSolverConfig.model_construct(solver="dp3_gaincal", niter=50, tol=1e-6, crosspol=True, phase_only=False), VisibilityFilterConfig.model_construct(), PlotConfig.model_construct())

    # Ignoring other tasks like plots since task_manager is a singleton
    print(dask.compute(_up_out.gaintable, optimize_graph=True))

t()